In [1]:
import torch
 
print(f"PyTorch version {torch.__version__}")
 
if torch.cuda.is_available():
    print(f"CUDA/ROCm GPU: {torch.cuda.get_device_name(0)}")
 
elif torch.xpu.is_available():
    print(f"Intel GPU: {torch.xpu.get_device_name(0)}")
 
elif torch.backends.mps.is_available():
    print("Apple Silicon GPU")
 
else:
    print("Only CPU")
 

PyTorch version 2.14.0+cpu
Only CPU


# 2.4 Preparing input texts for LLMs

In [2]:
from reasoning_from_scratch.qwen3 import download_qwen3_small
download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

In [3]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
 
tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

In [4]:
prompt = "Explain large language models."
input_token_ids_list = tokenizer.encode(prompt)

In [5]:
text = tokenizer.decode(input_token_ids_list)
print(text)

Explain large language models.


In [6]:
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

840 --> Ex
20772 --> plain
3460 -->  large
4128 -->  language
4119 -->  models
13 --> .


In [7]:
prompt = "kij##%$#@!@#$%^&*()_+|}{\":?><,./;'[]\\=-`~"
input_token_ids_list = tokenizer.encode(prompt)
for i in input_token_ids_list:
    print(f"{i} --> {tokenizer.decode([i])}")

74 --> k
3172 --> ij
565 --> ##
4 --> %
3 --> $
82072 --> #@
0 --> !
31 --> @
48077 --> #$
45899 --> %^
5 --> &
9 --> *
368 --> ()
62 --> _
10 --> +
91 --> |
15170 --> }{
788 --> ":
55317 --> ?><
11 --> ,
1725 --> ./
35994 --> ;'
1294 --> []
59 --> \
10829 --> =-
63 --> `
93 --> ~


# 2.5 Loading pretrained models

In [8]:
def get_device(enable_tensor_cores=True):
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using NVIDIA CUDA GPU")

        if enable_tensor_cores:
            major, minor = map(int, torch.__version__.split(".")[:2])
            if (major, minor) >= (2, 9):
                torch.backends.cuda.matmul.fp32_precision = "tf32"
                torch.backends.cudnn.conv.fp32_precision = "tf32"
            else:
                torch.backends.cuda.matmul.allow_tf32 = True
                torch.backends.cudnn.allow_tf32 = True

    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS)")

    elif torch.xpu.is_available():
        device = torch.device("xpu")
        print("Using Intel GPU")

    else:
        device = torch.device("cpu")
        print("Using CPU")

    return device

In [9]:
device = get_device()

Using CPU


In [10]:
download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

qwen3-0.6B-base.pth: 100% (1433 MiB / 1433 MiB)


In [11]:
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B

model_path = Path("qwen3") / "qwen3-0.6B-base.pth"
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)